In [2]:
import pandas as pd

file_path = r"C:\Users\munes\Dropbox\PC\Downloads\Dissertation MS\Phishing_Email.csv"

df = pd.read_csv(file_path)

print(df.head())
print(df.shape)
print(df.columns)

   Unnamed: 0                                         Email Text  \
0           0  re : 6 . 1100 , disc : uniformitarianism , re ...   
1           1  the other side of * galicismos * * galicismo *...   
2           2  re : equistar deal tickets are you still avail...   
3           3  \nHello I am your hot lil horny toy.\n    I am...   
4           4  software at incredibly low prices ( 86 % lower...   

       Email Type  
0      Safe Email  
1      Safe Email  
2      Safe Email  
3  Phishing Email  
4  Phishing Email  
(18650, 3)
Index(['Unnamed: 0', 'Email Text', 'Email Type'], dtype='object')


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18650 entries, 0 to 18649
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  18650 non-null  int64 
 1   Email Text  18634 non-null  object
 2   Email Type  18650 non-null  object
dtypes: int64(1), object(2)
memory usage: 437.2+ KB


In [4]:
df.isnull().sum()

Unnamed: 0     0
Email Text    16
Email Type     0
dtype: int64

In [5]:
df["Email Type"].value_counts()

Email Type
Safe Email        11322
Phishing Email     7328
Name: count, dtype: int64

In [6]:
df = df.drop(columns=["Unnamed: 0"])

df = df.rename(columns={
    "Email Text": "text",
    "Email Type": "label"
})

print(df.head())
print(df.columns)

                                                text           label
0  re : 6 . 1100 , disc : uniformitarianism , re ...      Safe Email
1  the other side of * galicismos * * galicismo *...      Safe Email
2  re : equistar deal tickets are you still avail...      Safe Email
3  \nHello I am your hot lil horny toy.\n    I am...  Phishing Email
4  software at incredibly low prices ( 86 % lower...  Phishing Email
Index(['text', 'label'], dtype='object')


In [7]:
df = df.dropna(subset=["text", "label"])

df["text"] = df["text"].astype(str)

df = df[df["text"].str.strip() != ""]
df = df[df["text"].str.lower().str.strip() != "empty"]

df = df.drop_duplicates(subset=["text"])

print(df.shape)
print(df["label"].value_counts())

(17535, 2)
label
Safe Email        10979
Phishing Email     6556
Name: count, dtype: int64


In [8]:
df["label_num"] = df["label"].map({
    "Safe Email": 0,
    "Phishing Email": 1
})

print(df[["label", "label_num"]].head())
print(df["label_num"].value_counts())

            label  label_num
0      Safe Email          0
1      Safe Email          0
2      Safe Email          0
3  Phishing Email          1
4  Phishing Email          1
label_num
0    10979
1     6556
Name: count, dtype: int64


In [9]:
df["text_length"] = df["text"].apply(len)
df["word_count"] = df["text"].apply(lambda x: len(x.split()))

print(df[["text_length", "word_count"]].head())

   text_length  word_count
0         1030         230
1          479          91
2         1245         305
3          688          96
4          441          91


In [10]:
import re

def count_urls(text):
    urls = re.findall(r"http[s]?://\S+|www\.\S+", text)
    return len(urls)

df["url_count"] = df["text"].apply(count_urls)
df["has_url"] = df["url_count"].apply(lambda x: 1 if x > 0 else 0)

print(df[["url_count", "has_url"]].head())

   url_count  has_url
0          0        0
1          0        0
2          0        0
3          1        1
4          0        0


In [11]:
suspicious_words = [
    "urgent", "verify", "password", "account", "blocked",
    "suspended", "login", "click", "confirm", "bank",
    "security", "update", "limited", "immediately", "winner",
    "free", "payment", "invoice", "refund"
]

def find_suspicious_words(text):
    text_lower = text.lower()
    found_words = []

    for word in suspicious_words:
        if word in text_lower:
            found_words.append(word)

    return len(found_words), ", ".join(found_words)

df[["suspicious_word_count", "risk_keyword_found"]] = df["text"].apply(
    lambda x: pd.Series(find_suspicious_words(x))
)

print(df[["suspicious_word_count", "risk_keyword_found"]].head())

   suspicious_word_count risk_keyword_found
0                      0                   
1                      0                   
2                      0                   
3                      1               free
4                      0                   


In [12]:
def clean_email_text(text):
    text = text.lower()
    text = re.sub(r"http[s]?://\S+|www\.\S+", " urltoken ", text)
    text = re.sub(r"\S+@\S+", " emailtoken ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_email_text)

print(df[["text", "clean_text"]].head())

                                                text  \
0  re : 6 . 1100 , disc : uniformitarianism , re ...   
1  the other side of * galicismos * * galicismo *...   
2  re : equistar deal tickets are you still avail...   
3  \nHello I am your hot lil horny toy.\n    I am...   
4  software at incredibly low prices ( 86 % lower...   

                                          clean_text  
0  re disc uniformitarianism re sex lang dick hud...  
1  the other side of galicismos galicismo is a sp...  
2  re equistar deal tickets are you still availab...  
3  hello i am your hot lil horny toy i am the one...  
4  software at incredibly low prices lower draper...  


In [13]:
output_path = r"C:\Users\munes\Dropbox\PC\Downloads\Dissertation MS\cleaned_phishing_email_dataset.csv"

df.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully.")
print(output_path)

Cleaned dataset saved successfully.
C:\Users\munes\Dropbox\PC\Downloads\Dissertation MS\cleaned_phishing_email_dataset.csv


In [14]:
print(df.shape)
print(df["label"].value_counts())

(17535, 10)
label
Safe Email        10979
Phishing Email     6556
Name: count, dtype: int64
